<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 강화학습·에이전트 · 06. DDPG

## Continuous Control with Deep Reinforcement Learning

- **원 논문:** [Continuous Control with Deep Reinforcement Learning](https://arxiv.org/abs/1509.02971)
- **저자 / 발표:** Timothy P. Lillicrap et al. · ICLR (2016)
- **난이도 / 예상 시간:** 고급 · 약 65분
- **선수 지식:** actor-critic, deterministic policy gradient, target networks
- **로컬 학습 데이터:** `data/field_curriculum/gridworld.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 로컬 상태에서 연속 action proxy를 만들고 deterministic actor, critic, soft target update를 학습한다.

**축소하거나 생략한 부분:** 논문은 MuJoCo 연속제어와 replay/OU noise를 사용한다. 축소판은 offline_action_noise로 만든 작은 continuous replay에서 Algorithm 1을 보존한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2, Eq. (3) | deterministic Bellman equation | Q(s,a)=E[r+γQ(s',mu(s'))] target을 구성한다. |
| §3, Eq. (4)–(6) | critic TD loss and deterministic policy gradient | critic을 회귀하고 actor는 Q를 action 방향으로 최대화한다. |
| §3, Algorithm 1 | replay, exploration noise, soft target update | prepared noise와 tau interpolation을 구현한다. |

## 핵심 재현

        Eq. (6)의 deterministic policy gradient는 actor action을 critic에 넣어 $Q(s,\mu(s))$를
        최대화합니다. `offline_action_noise`로 만든 로컬 continuous replay와 soft target을 사용합니다.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(461)
np.random.seed(461)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Tiny scalar/table loops may be faster on CPU even when a GPU exists;
# AI_LAB_DEVICE=cpu remains the low-overhead override.
DATA_PATH = Path("data/field_curriculum/gridworld.json")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
config = json.loads(DATA_PATH.read_text(encoding="utf-8"))
required = {
    "seed",
    "size",
    "start_state",
    "terminal_state",
    "gamma",
    "transitions",
    "bandit_means",
    "offline_action_noise",
}
assert required.issubset(config), f"JSON 키 불일치: {sorted(config)}"
rows = config["transitions"]
states = torch.tensor([int(r["state"]) for r in rows], dtype=torch.long)
actions = torch.tensor([int(r["action"]) for r in rows], dtype=torch.long)
rewards = torch.tensor([float(r["reward"]) for r in rows], dtype=torch.float32)
next_states = torch.tensor([int(r["next_state"]) for r in rows], dtype=torch.long)
dones = torch.tensor([float(r["done"]) for r in rows], dtype=torch.float32)
n_states = int(max(states.max(), next_states.max()).item()) + 1
n_actions = int(actions.max().item()) + 1
gamma = float(config["gamma"])
start_state, terminal_state = int(config["start_state"]), int(config["terminal_state"])
transition_table = {
    (int(r["state"]), int(r["action"])): (
        int(r["next_state"]),
        float(r["reward"]),
        bool(r["done"]),
    )
    for r in rows
}


def encode(s):
    return F.one_hot(torch.as_tensor(s, dtype=torch.long), n_states).float()


def encode_device(s):
    return encode(s).to(DEVICE)


def env_step(state, action):
    return transition_table[(int(state), int(action))]


def value_iteration(iterations=300):
    q = torch.zeros(n_states, n_actions)
    for _ in range(iterations):
        old = q.clone()
        for (s, a), (ns, r, done) in transition_table.items():
            q[s, a] = r if done else r + gamma * old[ns].max()
    return q


q_star = value_iteration()
assert torch.isfinite(q_star).all() and len(rows) > 0
print(ACCELERATOR.summary())
print(
    "RL environment, replay metadata, and tree search stay on CPU; "
    "neural batches use DEVICE."
)
print(
    f"local MDP: states={n_states}, actions={n_actions}, "
    f"transitions={len(rows)}, gamma={gamma}"
)

## 포트폴리오 구현 설계: 수식 → 에이전트 책임 → 검증

논문의 핵심 update를 실행 가능한 에이전트의 `forward`, `loss`, `update`,
`evaluate`로 나눕니다. 아래 식의 선택·평가·gradient 경계를 메서드 본문에서
한 줄씩 추적하세요.

$$
\mathcal L_Q=\mathbb E[(r+\gamma Q'(s',\mu'(s'))-Q(s,a))^2],\qquad J_\mu=-\mathbb E[Q(s,\mu(s))]
$$

- **기호와 역할:** $\mu$는 deterministic actor, $Q$는 critic, prime 표기는 soft target network입니다.
- **shape/state 계약:** `state/action [B, 1] → concat [B, 2] → Q [B, 1]`
- **논문 위치:** `§2, Eq. (3)`의 **deterministic Bellman equation**
- **코드 Task:** critic Bellman loss, actor Q maximization과 Polyak soft update를 구현합니다.
- **학습·평가 흐름:** continuous replay → critic update → actor update → target interpolation → action 평가
- **원문 대비 한계:** 논문은 MuJoCo 연속제어와 replay/OU noise를 사용한다. 축소판은 offline_action_noise로 만든 작은 continuous replay에서 Algorithm 1을 보존한다.
- **구현 이유:** target, gradient, optimizer 책임을 Agent 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 Bellman target, policy objective, search score 같은 핵심 계산을
불투명한 trainer 뒤에 감추지 않습니다. 실습본에서는 공개 API를 유지하면서 TODO를
채우고, 각 메서드의 입력 shape와 gradient가 끊기는 위치를 설명하세요.

In [ ]:
class DeterministicPolicyAgent(nn.Module):
    """DDPG의 핵심 학습·평가 경로. 입출력 계약: state/action [B, 1] → concat [B, 2] → Q [B, 1]."""

    def __init__(
        self,
        actor_network,
        critic_network,
        target_actor,
        target_critic,
        actor_optimizer,
        critic_optimizer,
        discount,
        tau=0.02,
    ):
        """network, optimizer, target 또는 table 상태를 명시적으로 저장한다."""
        super().__init__()
        self.actor_network = actor_network
        self.critic_network = critic_network
        self.target_actor = target_actor
        self.target_critic = target_critic
        self.actor_optimizer = actor_optimizer
        self.critic_optimizer = critic_optimizer
        self.discount = discount
        self.tau = tau

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"action_dim": 1, "gamma": self.discount, "tau": self.tau}

    def forward(self, state):
        """policy/value/Q 순전파를 계산한다. shape 계약: state/action [B, 1] → concat [B, 2] → Q
        [B, 1]."""
        action = self.actor_network(state)
        q_value = self.critic_network(state, action)
        return (action, q_value)

    def loss(self, state, action, reward, next_state, done):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        with torch.no_grad():
            next_action = self.target_actor(next_state)
            target_q = self.target_critic(next_state, next_action)
            target_q = reward + self.discount * (1.0 - done) * target_q
        critic_loss = F.mse_loss(self.critic_network(state, action), target_q)
        policy_action = self.actor_network(state)
        actor_loss = -self.critic_network(state, policy_action).mean()
        return (critic_loss, actor_loss)

    def update(self, state, action, reward, next_state, done):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        with torch.no_grad():
            next_action = self.target_actor(next_state)
            target_q = self.target_critic(next_state, next_action)
            target_q = reward + self.discount * (1.0 - done) * target_q
        critic_loss = F.mse_loss(self.critic_network(state, action), target_q)
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        policy_action = self.actor_network(state)
        actor_loss = -self.critic_network(state, policy_action).mean()
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()
        self._soft_update(self.target_actor, self.actor_network)
        self._soft_update(self.target_critic, self.critic_network)
        return (float(critic_loss.detach()), float(actor_loss.detach()))

    def _soft_update(self, target_network, source_network):
        """Polyak 평균으로 target parameter를 제자리 갱신한다."""
        with torch.no_grad():
            for target_parameter, source_parameter in zip(
                target_network.parameters(), source_network.parameters()
            ):
                target_parameter.mul_(1.0 - self.tau)
                target_parameter.add_(source_parameter, alpha=self.tau)

    def evaluate(self, state, target_action):
        """no_grad 경로에서 정책 또는 value 품질 지표 dict를 반환한다."""
        with torch.no_grad():
            action, q_value = self.forward(state)
            action_mse = F.mse_loss(action, target_action)
        return {"action_mse": float(action_mse), "mean_q": float(q_value.mean())}

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§3, Algorithm 1` — **replay, exploration noise, soft target update**
- **구현 이유:** prepared noise와 tau interpolation을 구현한다.
- **읽을 코드:** 아래 `implementation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 1), nn.Tanh()
        )

    def forward(self, s):
        return self.net(s)


class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 48), nn.ReLU(), nn.Linear(48, 1))

    def forward(self, s, a):
        return self.net(torch.cat([s, a], 1))


def soft_update(target, source, tau):
    with torch.no_grad():
        for tp, sp in zip(target.parameters(), source.parameters()):
            tp.mul_(1 - tau).add_(sp, alpha=tau)


actor, critic, actor_t, critic_t = ACCELERATOR.move(
    Actor(), Critic(), Actor(), Critic()
)
actor_t.load_state_dict(actor.state_dict())
critic_t.load_state_dict(critic.state_dict())

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§3, Eq. (4)–(6)` — **critic TD loss and deterministic policy gradient**
- **구현 이유:** critic을 회귀하고 actor는 Q를 action 방향으로 최대화한다.
- **읽을 코드:** 아래 `training` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
state_grid = torch.arange(n_states).float()[:, None] / max(1, n_states - 1) * 2 - 1
best = q_star.argmax(1).float()
target_action = best[:, None] / max(1, n_actions - 1) * 2 - 1
repeat = 40
replay_s = state_grid.repeat_interleave(repeat, 0)
replay_target = target_action.repeat_interleave(repeat, 0)
local_noise = torch.tensor(config["offline_action_noise"], dtype=torch.float32)
local_noise = local_noise.repeat(math.ceil(len(replay_target) / len(local_noise)))[
    : len(replay_target), None
]
action_sweep = torch.linspace(-1, 1, repeat).repeat(n_states)[:, None]
replay_a = (action_sweep + local_noise).clamp(-1, 1)
replay_r = -(replay_a - replay_target).square()
replay_ns = replay_s.clone()
replay_done = torch.ones_like(replay_r)
(
    state_grid,
    target_action,
    replay_s,
    replay_a,
    replay_r,
    replay_ns,
    replay_done,
) = ACCELERATOR.move(
    state_grid,
    target_action,
    replay_s,
    replay_a,
    replay_r,
    replay_ns,
    replay_done,
)
oc = torch.optim.Adam(critic.parameters(), lr=0.006)
oa = torch.optim.Adam(actor.parameters(), lr=0.004)
gen = torch.Generator().manual_seed(463)
history = []

portfolio_agent = DeterministicPolicyAgent(
    actor,
    critic,
    actor_t,
    critic_t,
    oa,
    oc,
    gamma,
)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in actor.parameters()]
)
for _ in range(320):
    index = torch.randint(len(replay_s), (64,), generator=gen).to(DEVICE)
    history.append(
        portfolio_agent.update(
            replay_s[index],
            replay_a[index],
            replay_r[index],
            replay_ns[index],
            replay_done[index],
        )
    )
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in actor.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
assert parameter_delta > 0.0 and np.isfinite(history).all()

### 단계 4. 평가·시각화

- **논문의 어느 부분인가:** `§2, Eq. (3)` — **deterministic Bellman equation**
- **구현 이유:** Q(s,a)=E[r+γQ(s',mu(s'))] target을 구성한다.
- **읽을 코드:** 아래 `evaluation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
portfolio_metrics = portfolio_agent.evaluate(state_grid, target_action)
assert portfolio_metrics

with torch.no_grad():
    learned_action = actor(state_grid)
    action_mse = float(F.mse_loss(learned_action, target_action))
    target_gap = sum(
        float((p - q).square().sum())
        for p, q in zip(actor.parameters(), actor_t.parameters())
    )
print(f"actor target MSE={action_mse:.4f}, online-target gap={target_gap:.6f}")
assert action_mse < 0.6 and math.isfinite(target_gap)
h = np.asarray(history)
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(target_action.detach().cpu(), label="optimal proxy")
axes[0].plot(learned_action.detach().cpu(), label="actor")
axes[0].legend()
axes[1].plot(h[:, 0], label="critic")
axes[1].plot(h[:, 1], label="actor")
axes[1].legend()
fig.tight_layout()
plt.show()

deterministic policy gradient는 현재 policy가 선택하는 action에서 Q의 action gradient를 따라야 합니다. replay action은 critic 회귀에 쓰지만 actor의 결정 경로가 아닙니다.

### 단계 5. 통합 Agent가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§2, Eq. (3)`의 **deterministic Bellman equation**
- **핵심 식:**

  $$
  \mathcal L_Q=\mathbb E[(r+\gamma Q'(s',\mu'(s'))-Q(s,a))^2],\qquad J_\mu=-\mathbb E[Q(s,\mu(s))]
  $$

- **입출력 shape:** `state/action [B, 1] → concat [B, 2] → Q [B, 1]`
- **구현 이유:** 앞 셀은 target, objective, search를 작은 연산으로
  분해해 확인하고, 이 셀은 같은 구성 요소를
  `DeterministicPolicyAgent`에 주입해 최종 학습·평가 경로를
  하나로 묶습니다. Q(s,a)=E[r+γQ(s',mu(s'))] target을 구성한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 정책·가치 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_agent.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_agent).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 MuJoCo 연속제어와 replay/OU noise를 사용한다. 축소판은 offline_action_noise로 만든 작은 continuous replay에서 Algorithm 1을 보존한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.